# Preprocessing

In [ ]:
import pandas as pd
df=pd.read_csv("datasets/hotel_raw.csv")

In [2]:
df.head(1)

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date,city
0,Resort Hotel - Chandigarh,0,342,2024,July,30,27,0,0,2,...,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2024-07-27 22:16:40.916332324,Chandigarh


In [3]:
df["city"] = df["hotel"].str.split(" - ").str[-1]

In [4]:
df["city"].head()

0    Chandigarh
1        Mumbai
2         Delhi
3       Kolkata
4       Lucknow
Name: city, dtype: object

In [5]:
df["arrival_date"] = pd.to_datetime(
    df["arrival_date_year"].astype(str) + "-" +
    df["arrival_date_month"] + "-" +
    df["arrival_date_day_of_month"].astype(str),
    format="%Y-%B-%d"
)

# format as YYYY-MM-DD
df["arrival_date"] = df["arrival_date"].dt.strftime("%Y-%m-%d")

In [6]:
df["arrival_date"].head()

0    2024-07-27
1    2024-04-28
2    2024-09-10
3    2024-08-14
4    2024-09-14
Name: arrival_date, dtype: object

In [7]:
import pandas as pd
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
import time

In [8]:
cities = df["city"].dropna().unique()
geolocator = Nominatim(user_agent="hotel_location_app_v1", timeout=10)
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1.5)

city_coords = {}

print(f"Finding coordinates for {len(cities)} cities...")

for city in cities:
    try:
        location = geocode(city)
        if location:
            city_coords[city] = (location.latitude, location.longitude)
            print(f"Found: {city} -> {location.latitude}, {location.longitude}")
        else:
            city_coords[city] = (None, None)
            print(f"Not found: {city}")
    except Exception as e:
        print(f"Error occurred ({city}): {e}")
        city_coords[city] = (None, None)
        time.sleep(1)

df["lat"] = df["city"].map(lambda x: city_coords.get(x, (None, None))[0])
df["lon"] = df["city"].map(lambda x: city_coords.get(x, (None, None))[1])

print("Operation completed!")

Finding coordinates for 15 cities...
Found: Chandigarh -> 30.7334421, 76.7797143
Found: Mumbai -> 19.054999, 72.8692035
Found: Delhi -> 28.6138954, 77.2090057
Found: Kolkata -> 22.5726459, 88.3638953
Found: Lucknow -> 26.8381, 80.9346001
Found: Indore -> 22.7203616, 75.8681996
Found: Ahmedabad -> 23.0215374, 72.5800568
Found: Pune -> 18.5213738, 73.8545071
Found: Chennai -> 13.0836939, 80.270186
Found: Bhopal -> 23.2584857, 77.401989
Found: Hyderabad -> 17.360589, 78.4740613
Found: Kochi -> 9.9679032, 76.2444378
Found: Bangalore -> 12.9767936, 77.590082
Found: Jaipur -> 26.9154576, 75.8189817
Found: Goa -> 15.3004543, 74.0855134
Operation completed!


In [9]:
df.head()

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date,city,arrival_date,lat,lon
0,Resort Hotel - Chandigarh,0,342,2024,July,30,27,0,0,2,...,Transient,0.0,0,0,Check-Out,2024-07-27 22:16:40.916332324,Chandigarh,2024-07-27,30.733442,76.779714
1,Resort Hotel - Mumbai,0,737,2024,April,17,28,0,0,2,...,Transient,0.0,0,0,Check-Out,2024-04-28 21:56:21.507509066,Mumbai,2024-04-28,19.054999,72.869203
2,Resort Hotel - Delhi,0,7,2024,September,37,10,0,1,1,...,Transient,75.0,0,0,Check-Out,2024-09-10 03:46:25.734029096,Delhi,2024-09-10,28.613895,77.209006
3,Resort Hotel - Kolkata,0,13,2024,August,33,14,0,1,1,...,Transient,75.0,0,0,Check-Out,2024-08-14 18:07:10.049669568,Kolkata,2024-08-14,22.572646,88.363895
4,Resort Hotel - Lucknow,0,14,2024,September,37,14,0,2,2,...,Transient,98.0,0,1,Check-Out,2024-09-14 14:27:32.473846000,Lucknow,2024-09-14,26.838100,80.934600


In [10]:
import openmeteo_requests
import pandas as pd
import requests_cache
from retry_requests import retry
import time

# ---------------------------------------------
# Open-Meteo setup
# ---------------------------------------------
cache_session = requests_cache.CachedSession(".cache", expire_after=-1)
retry_session = retry(cache_session, retries=5, backoff_factor=0.2)
openmeteo = openmeteo_requests.Client(session=retry_session)

url = "https://archive-api.open-meteo.com/v1/archive"

# ---------------------------------------------
# Create columns
# ---------------------------------------------
df["temperature_2m_mean"] = None
df["rain_sum"] = None
df["snowfall_sum"] = None
df["sunrise"] = None
df["sunset"] = None

# ---------------------------------------------
# Loop rows
# ---------------------------------------------
for idx, row in df.iterrows():

    if pd.isna(row["lat"]) or pd.isna(row["lon"]) or pd.isna(row["arrival_date"]):
        continue

    params = {
        "latitude": row["lat"],
        "longitude": row["lon"],
        "start_date": row["arrival_date"],
        "end_date": row["arrival_date"],
        "daily": [
            "sunrise",
            "sunset",
            "rain_sum",
            "snowfall_sum",
            "temperature_2m_mean"
        ],
        "timezone": "GMT"
    }

    try:
        responses = openmeteo.weather_api(url, params=params)
        daily = responses[0].Daily()

        df.at[idx, "sunrise"] = daily.Variables(0).ValuesInt64AsNumpy()[0]
        df.at[idx, "sunset"] = daily.Variables(1).ValuesInt64AsNumpy()[0]
        df.at[idx, "rain_sum"] = daily.Variables(2).ValuesAsNumpy()[0]
        df.at[idx, "snowfall_sum"] = daily.Variables(3).ValuesAsNumpy()[0]
        df.at[idx, "temperature_2m_mean"] = daily.Variables(4).ValuesAsNumpy()[0]

        print(f"✔ Weather added: {row['city']} | {row['arrival_date']}", flush=True)

       
        if idx % 100 == 0:
            df.to_csv(r"C:\Users\Elvin Aliyev\Downloads\hotel_backup.csv", index=False)
            print(f"💾 Backup saved at row {idx}", flush=True)

        time.sleep(0.2)

    except Exception as e:
        print(f"❌ Error for {row['city']} ({row['arrival_date']}): {e}", flush=True)


df.to_csv(r"C:\Users\Elvin Aliyev\Downloads\hotel_final.csv", index=False)
print("✅ Weather data appended successfully!")

✔ Weather added: Chandigarh | 2024-07-27
💾 Backup saved at row 0
✔ Weather added: Mumbai | 2024-04-28
✔ Weather added: Delhi | 2024-09-10
✔ Weather added: Kolkata | 2024-08-14
✔ Weather added: Lucknow | 2024-09-14
✔ Weather added: Delhi | 2024-10-23
✔ Weather added: Indore | 2024-01-07
✔ Weather added: Ahmedabad | 2024-08-08
✔ Weather added: Delhi | 2024-08-25
✔ Weather added: Pune | 2024-12-18
✔ Weather added: Chennai | 2024-08-24
✔ Weather added: Chennai | 2024-03-27
✔ Weather added: Bhopal | 2024-03-02
✔ Weather added: Lucknow | 2024-06-09
✔ Weather added: Hyderabad | 2024-06-15
✔ Weather added: Mumbai | 2024-06-24
✔ Weather added: Kochi | 2024-01-18
✔ Weather added: Chennai | 2024-12-17
✔ Weather added: Mumbai | 2024-04-17
✔ Weather added: Delhi | 2024-11-11
✔ Weather added: Pune | 2024-11-23
✔ Weather added: Bangalore | 2024-01-13
✔ Weather added: Bhopal | 2024-07-04
✔ Weather added: Chennai | 2024-03-22
✔ Weather added: Ahmedabad | 2024-02-25
✔ Weather added: Pune | 2024-07-24
✔ 

In [12]:
df.head()

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,arrival_date,lat,lon,temperature_2m_mean,rain_sum,snowfall_sum,sunrise,sunset,sunrise_dt,sunset_dt
0,Resort Hotel - Chandigarh,0,342,2024,July,30,27,0,0,2,...,2024-07-27,30.733442,76.779714,31.449999,3.3,0.0,1722038852,1722088254,2024-07-27 00:07:32+00:00,2024-07-27 13:50:54+00:00
1,Resort Hotel - Mumbai,0,737,2024,April,17,28,0,0,2,...,2024-04-28,19.054999,72.869203,33.218746,0.0,0.0,1714264928,1714311000,2024-04-28 00:42:08+00:00,2024-04-28 13:30:00+00:00
2,Resort Hotel - Delhi,0,7,2024,September,37,10,0,1,1,...,2024-09-10,28.613895,77.209006,27.170832,9.6,0.0,1725928440,1725973337,2024-09-10 00:34:00+00:00,2024-09-10 13:02:17+00:00
3,Resort Hotel - Kolkata,0,13,2024,August,33,14,0,1,1,...,2024-08-14,22.572646,88.363895,28.13125,39.999996,0.0,1723592606,1723639153,2024-08-13 23:43:26+00:00,2024-08-14 12:39:13+00:00
4,Resort Hotel - Lucknow,0,14,2024,September,37,14,0,2,2,...,2024-09-14,26.838100,80.934600,26.945833,0.1,0.0,1726273292,1726317720,2024-09-14 00:21:32+00:00,2024-09-14 12:42:00+00:00


In [2]:
import pandas as pd
df=pd.read_csv('hotel_final.csv')